# Defensive Security Proxy: Evaluation Analysis and Visualizations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cntrvsy/masters/blob/main/dataset/results_analysis.ipynb)

This notebook provides the interactive visual evaluations comparing the performance of the **Fine-Tuned Defensive Proxy model** (`qwen2.5-3b-instruct_skills_security_proxy`) against the **Vanilla Baseline model** (`qwen2.5-vl-3b-instruct`) on our holdout evaluation datasets.

## Visualizations Included:
1. **Security Vulnerability Matrix**: Attack Success Rate (ASR-valid) and Benign False-Alarm Rates.
2. **Format Integrity Pass Rate**: Percentage of parsable/valid YAML responses.
3. **Token Compression Ratio Distribution**: Box plots contrasting the efficiency metrics.
4. **Output length vs. Input length**: Scatter plot visualizing formatting truncation vs. proper sanitization scaling.

## Data Source:
The evaluation datasets are loaded dynamically from the `cntrvsy/masters` GitHub repository raw endpoints.

In [1]:
# Install dependencies if running in a fresh environment like Google Colab
# %pip install pandas altair vega_datasets

In [2]:
import pandas as pd
import json
import urllib.request
import altair as alt

print("Libraries imported successfully.")

Libraries imported successfully.


## 1. Load Evaluation Results from GitHub
We load the datasets directly from GitHub raw URLs using `pandas.read_json` with `lines=True`.

In [3]:
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/cntrvsy/masters/main/dataset/results/"

# Paths for Fine-Tuned Model
FT_URLS = {
    "Base (Standard Attacks)": GITHUB_RAW_BASE + "qwen2.5-3b-instruct_skills_security_proxy/evaluation_lmstudio_standard_base_results.jsonl",
    "Enhanced (Jailbreak Attacks)": GITHUB_RAW_BASE + "qwen2.5-3b-instruct_skills_security_proxy/evaluation_lmstudio_standard_enhanced_results.jsonl",
    "Benign (Hard Negatives)": GITHUB_RAW_BASE + "qwen2.5-3b-instruct_skills_security_proxy/evaluation_lmstudio_standard_benign_results.jsonl"
}

# Paths for Vanilla Baseline Model
BASE_URLS = {
    "Base (Standard Attacks)": GITHUB_RAW_BASE + "qwen2.5-vl-3b-instruct/evaluation_lmstudio_standard_base_results.jsonl",
    "Enhanced (Jailbreak Attacks)": GITHUB_RAW_BASE + "qwen2.5-vl-3b-instruct/evaluation_lmstudio_standard_enhanced_results.jsonl",
    "Benign (Hard Negatives)": GITHUB_RAW_BASE + "qwen2.5-vl-3b-instruct/evaluation_lmstudio_standard_benign_results.jsonl"
}

def load_remote_jsonl(urls_dict, model_name):
    dfs = []
    for setting, url in urls_dict.items():
        try:
            df = pd.read_json(url, lines=True)
            df['setting'] = setting
            df['Model'] = model_name
            dfs.append(df)
            print(f"Loaded {len(df)} records for {model_name} ({setting})")
        except Exception as e:
            print(f"Failed to load {url}: {e}")
            print("Please verify that the repository is public and paths are correct.")
    if dfs:
        return pd.concat(dfs, ignore_index=True)
    return pd.DataFrame()

print("Loading Fine-Tuned results...")
ft_df = load_remote_jsonl(FT_URLS, "Fine-Tuned Proxy")

print("\nLoading Vanilla Baseline results...")
base_df = load_remote_jsonl(BASE_URLS, "Vanilla Baseline")

full_df = pd.concat([ft_df, base_df], ignore_index=True)
print(f"\nCombined Dataset: {len(full_df)} total evaluation records loaded.")

Loading Fine-Tuned results...
Loaded 554 records for Fine-Tuned Proxy (Base (Standard Attacks))
Loaded 554 records for Fine-Tuned Proxy (Enhanced (Jailbreak Attacks))
Loaded 90 records for Fine-Tuned Proxy (Benign (Hard Negatives))

Loading Vanilla Baseline results...
Loaded 554 records for Vanilla Baseline (Base (Standard Attacks))
Loaded 554 records for Vanilla Baseline (Enhanced (Jailbreak Attacks))
Loaded 90 records for Vanilla Baseline (Benign (Hard Negatives))

Combined Dataset: 2396 total evaluation records loaded.


## 2. Programmatic Verification & Combined Metrics Matrix
Let's compute the primary statistics programmatically to verify alignment with `results.md`.

In [4]:
summary_list = []
for model in ['Fine-Tuned Proxy', 'Vanilla Baseline']:
    model_df = full_df[full_df['Model'] == model]
    for setting in ['Base (Standard Attacks)', 'Enhanced (Jailbreak Attacks)', 'Benign (Hard Negatives)']:
        sub_df = model_df[model_df['setting'] == setting]
        if sub_df.empty:
            continue

        total = len(sub_df)
        valid_format = sub_df['is_valid_format'].sum()
        format_rate = (valid_format / total) * 100
        avg_compression = sub_df['compression_ratio'].mean()

        if setting == 'Benign (Hard Negatives)':
            # For Benign, false-alarm/mangle rate is 100 - format pass rate
            false_alarm = 100.0 - format_rate
            summary_list.append({
                'Model': model,
                'Setting': setting,
                'Format Pass Rate (%)': format_rate,
                'Attack Success Rate (%)': 0.0,
                'False-Alarm Rate (%)': false_alarm,
                'Avg. Compression': avg_compression,
                'Sample Count': total
            })
        else:
            # ASR-valid is evaluated among valid formats
            leaks = sub_df[sub_df['is_valid_format']]['attack_success'].sum()
            asr_valid = (leaks / valid_format * 100) if valid_format > 0 else 0.0
            summary_list.append({
                'Model': model,
                'Setting': setting,
                'Format Pass Rate (%)': format_rate,
                'Attack Success Rate (%)': asr_valid,
                'False-Alarm Rate (%)': 0.0,
                'Avg. Compression': avg_compression,
                'Sample Count': total
            })

summary_df = pd.DataFrame(summary_list)
summary_df

,Model,Setting,Format Pass Rate (%),Attack Success Rate (%),False-Alarm Rate (%),Avg. Compression,Sample Count
0,Fine-Tuned Proxy,Base (Standard Attacks),100.000000,0.000000,0.000000,1.393298,554
1,Fine-Tuned Proxy,Enhanced (Jailbreak Attacks),100.000000,0.000000,0.000000,1.528820,554
2,Fine-Tuned Proxy,Benign (Hard Negatives),100.000000,0.000000,0.000000,1.144890,90
3,Vanilla Baseline,Base (Standard Attacks),96.209386,3.377111,0.000000,4.692895,554
4,Vanilla Baseline,Enhanced (Jailbreak Attacks),96.750903,2.238806,0.000000,5.084446,554
5,Vanilla Baseline,Benign (Hard Negatives),96.666667,0.000000,3.333333,4.024103,90


## 3. Registering the Custom Theme & Styles
We customize Altair's global configuration to output clean, publishing-ready charts.

In [5]:
def custom_theme():
    return {
        'config': {
            'view': {
                'stroke': 'transparent',
                'width': 400,
                'height': 300
            },
            'axis': {
                'domain': False,
                'gridColor': '#f1f5f9',
                'gridDash': [4, 4],
                'labelColor': '#64748b',
                'labelFont': 'Inter, system-ui, sans-serif',
                'labelFontSize': 11,
                'titleColor': '#334155',
                'titleFont': 'Inter, system-ui, sans-serif',
                'titleFontSize': 12,
                'titleFontWeight': 600,
                'tickColor': 'transparent'
            },
            'legend': {
                'labelColor': '#64748b',
                'labelFont': 'Inter, system-ui, sans-serif',
                'labelFontSize': 11,
                'titleColor': '#334155',
                'titleFont': 'Inter, system-ui, sans-serif',
                'titleFontSize': 11,
                'titleFontWeight': 600,
                'orient': 'top-left'
            },
            'title': {
                'color': '#1e293b',
                'font': 'Inter, system-ui, sans-serif',
                'fontSize': 14,
                'fontWeight': 600,
                'anchor': 'start',
                'offset': 15
            }
        }
    }

# Register and enable the modern theme using Altair 5 theme registry
# We check version to use the appropriate API
try:
    alt.theme.register('custom_theme', custom_theme)
    alt.theme.enable('custom_theme')
except Exception:
    alt.themes.register('custom_theme', custom_theme)
    alt.themes.enable('custom_theme')

# Define a consistent color scheme: Fine-Tuned (Indigo) vs. Vanilla Baseline (Rose)
color_scale = alt.Scale(domain=['Fine-Tuned Proxy', 'Vanilla Baseline'], range=['#6366f1', '#f43f5e'])

print("Aesthetic theme configured.")

Aesthetic theme configured.


/tmp/ipykernel_5556/4129858840.py:49: AltairDeprecationWarning: 
Deprecated since `altair=5.5.0`. Use altair.theme instead.
Most cases require only the following change:

    # Deprecated
    alt.themes.enable('quartz')

    # Updated
    alt.theme.enable('quartz')

If your code registers a theme, make the following change:

    # Deprecated
    def custom_theme():
        return {'height': 400, 'width': 700}
    alt.themes.register('theme_name', custom_theme)
    alt.themes.enable('theme_name')

    # Updated
    @alt.theme.register('theme_name', enable=True)
    def custom_theme():
        return alt.theme.ThemeConfig(
            {'height': 400, 'width': 700}
        )

See the updated User Guide for further details:
    https://altair-viz.github.io/user_guide/api.html#theme
    https://altair-viz.github.io/user_guide/customization.html#chart-themes
  alt.themes.register('custom_theme', custom_theme)


## 4. Visualizations & Interpretations

### Diagram 1: Security Efficacy Matrix (ASR and False-Alarm rates)
This chart displays the security performance. We plot the **Attack Success Rate (ASR)** for base/enhanced datasets and the **Benign False-Alarm Rate** for the benign control dataset.

In [10]:
import altair as alt

# Filter out zero values for cleaner labeling if necessary
security_plot_data = pd.DataFrame([
    {'Model': r['Model'], 'Setting': r['Setting'], 'Metric': 'Attack Success Rate (ASR)', 'Value': r['Attack Success Rate (%)']}
    for r in summary_list if r['Setting'] != 'Benign (Hard Negatives)'
] + [
    {'Model': r['Model'], 'Setting': r['Setting'], 'Metric': 'Benign False-Alarm Rate', 'Value': r['False-Alarm Rate (%)']}
    for r in summary_list if r['Setting'] == 'Benign (Hard Negatives)'
])

base_chart = alt.Chart(security_plot_data).encode(
    x=alt.X('Model:N', title=None, axis=alt.Axis(labels=False, ticks=False)),
    y=alt.Y('Value:Q', title='Rate (%)', scale=alt.Scale(domain=[0, 4.5])),
    color=alt.Color('Model:N', scale=color_scale, legend=alt.Legend(orient='bottom', titleFontSize=12))
)

bars = base_chart.mark_bar(cornerRadius=4, size=60).encode(
    tooltip=['Model', 'Setting', 'Value']
)

text = base_chart.mark_text(
    align='center', baseline='bottom', dy=-5, fontWeight='bold'
).encode(text=alt.Text('Value:Q', format='.2f'))

chart_security_pro = (bars + text).facet(
    column=alt.Column('Setting:N', title='Dataset Evaluation Setting', header=alt.Header(labelFontSize=12, titleFontSize=13))
).properties(
    title=alt.TitleParams('Security Efficacy: Fine-Tuned vs. Vanilla Baseline', subtitle=['Lower is better (0% is ideal)'], anchor='start')
)

chart_security_pro

alt.FacetChart(...)

#### Security Discussion Points:
- **Defensive Safety Efficacy**: The Fine-Tuned Proxy achieves **0.00% ASR-valid** under both standard (Base) and jailbreak (Enhanced) datasets, confirming perfect safety. The Vanilla Baseline has a **3.38% ASR** under standard attacks and **2.24% ASR** under jailbreaks, representing active vulnerabilities.
- **False Alarm Zeroing**: The Fine-Tuned Proxy achieves **0.00% False-Alarm Rate** (mangle rate) on the Benign control dataset, indicating no functional degradation of benign system calls. The Vanilla Baseline incorrectly mangled **3.33%** of the benign requests.

### Diagram 2: Format Integrity Pass Rate
This chart visualizes the format pass rates. Upstream agents rely on strict YAML syntax compliance for parsing output instructions. Format failures lead to downstream crashes.

In [28]:
base_format = alt.Chart(summary_df).encode(
    y=alt.Y('Model:N', title=None, axis=alt.Axis(labels=False, ticks=False)),
    x=alt.X('Format Pass Rate (%):Q', title='Pass Rate (%)', scale=alt.Scale(domain=[90, 101])),
    color=alt.Color('Model:N', scale=color_scale, legend=alt.Legend(orient='bottom', title='Model'))
)

bars_format = base_format.mark_bar(cornerRadiusEnd=4).encode(
    y=alt.Y('Model:N', title=None, axis=alt.Axis(labels=False, ticks=False))
)

text_format = base_format.mark_text(
    align='left', baseline='middle', dx=5, fontWeight='bold', fontSize=11
).encode(text=alt.Text('Format Pass Rate (%):Q', format='.1f'))

chart_format_pro = (bars_format + text_format).properties(
    width=220,
    height=80
).facet(
    facet=alt.Facet('Setting:N', title=None, header=alt.Header(labelFontSize=12, labelFontWeight='bold')),
    columns=2
).properties(
    title=alt.TitleParams(
        'Format Integrity: YAML Parsing Success',
        subtitle=['Percentage of responses passing strict schema validation'],
        anchor='start'
    )
).configure_view(
    strokeWidth=0
).configure_axis(
    grid=True, gridOpacity=0.3
)

chart_format_pro

alt.FacetChart(...)

#### Format Discussion Points:
- **Alignment Generalization**: The Fine-Tuned Proxy achieves a perfect **100% Format Integrity** across all evaluation runs. This is a critical finding, demonstrating that alignment training successfully taught the model the YAML schema structure as a hard grammar rule.
- **Vanilla Bottleneck**: The Vanilla Baseline model regularly outputs syntactically incorrect/incomplete YAML blocks, failing format checks **3.25% - 3.79%** of the time. In a production environment, this introduces critical operational failures.

### Diagram 3: Token Compression Ratio Distribution
We use a Box Plot to show the token compression ratio ($Ratio = \frac{Input \, Tokens}{Output \, Tokens}$) distributions. This helps show the variance in content reduction.

In [18]:
rule_baseline = alt.Chart(pd.DataFrame({'y': [1.0]})).mark_rule(
    color='#94a3b8', strokeDash=[4, 4], size=1.5
).encode(y='y')

chart_compression = alt.Chart(full_df).mark_boxplot(
    extent='min-max', size=30, outliers=True, ticks=True
).encode(
    x=alt.X('setting:N', title=None, axis=alt.Axis(labelAngle=0, labelFontSize=11)),
    y=alt.Y('compression_ratio:Q', title='Compression Ratio (Input/Output)'),
    color=alt.Color('Model:N', scale=color_scale),
    xOffset='Model:N'
)

chart_compression_pro = (rule_baseline + chart_compression).properties(
    width=500, height=350,
    title=alt.TitleParams('Efficiency Analysis: Distribution of Token Compression Ratios', subtitle=['A ratio of 1.0 (dashed line) indicates original length preservation.'], anchor='start')
)

chart_compression_pro

alt.LayerChart(...)

#### Compression Discussion Points:
- **High Baseline Variance**: The Vanilla Baseline has highly variable, extremely high compression ratios (mostly grouped in the 4.0x–6.0x region). This signifies that the baseline model outputs very short messages relative to the input.
- **Predictable Proxy Scaling**: The Fine-Tuned Proxy sits in a much tighter, consistent range of 1.1x–1.6x. This indicates that it preserves input schemas and descriptions systematically rather than deleting blocks of content.

### Diagram 4: Output vs. Input Length Scaling (The Truncation Illusion)
This scatter plot maps the relationship between input token length (X-axis) and output token length (Y-axis), with regression trendlines for both models. The diagonal dashed line shows the 1:1 parity (Output = Input).

In [23]:
identity_line = alt.Chart(pd.DataFrame({'x': [0, 360], 'y': [0, 360]})).mark_line(
    color='#94a3b8', strokeDash=[4, 4], opacity=0.6
).encode(x='x', y='y')

# Tighten axes to data range (approx 0-550) to remove empty space
scatter = alt.Chart(full_df).mark_point(opacity=0.5, size=35, filled=True).encode(
    x=alt.X('input_tokens:Q', title='Input Tokens', scale=alt.Scale(domain=[0, 260])),
    y=alt.Y('output_tokens:Q', title='Output Tokens', scale=alt.Scale(domain=[0, 260])),
    color=alt.Color('Model:N', scale=color_scale),
    tooltip=['Model', 'input_tokens', 'output_tokens']
)

trend = scatter.transform_regression('input_tokens', 'output_tokens', groupby=['Model']).mark_line(size=4)

chart_scatter_pro = (identity_line + scatter + trend).properties(
    width=500, height=450,
    title=alt.TitleParams('The Truncation Illusion: Scaling Behavior', subtitle=['Dashed line: 1:1 Parity. Fine-tuned model scales correctly; Vanilla stays flat (truncation).'], anchor='start')
).interactive()

chart_scatter_pro

alt.LayerChart(...)

#### Truncation Illusion Discussion Points:
- **Over-Compression Failure (Laziness)**: The scatter plot reveals a major qualitative flaw in the Vanilla Baseline. Its output points cluster in a flat line at the bottom (between 15 and 35 output tokens) regardless of input size. The baseline achieves its high compression ratio by **dropping parameters, descriptions, and warnings**—truncating the instructions until they are useless.
- **Proportional Information Preservation**: The Fine-Tuned Proxy's trendline scales proportionally with the input size, remaining close to the 1:1 identity line. This proves that the fine-tuned model preserves the functional structure, only removing or replacing specific hostile components, which is the correct behavior for an agent security proxy.